In [9]:
!pip install underthesea

In [10]:
import os
import re
import json
import math
import logging
from collections import Counter, defaultdict
from typing import List, Dict, Any, Tuple

try:
    from underthesea import word_tokenize
except ImportError:
    print("Error")
    raise

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

INPUT_JSON_FILE = "chiaki_all_products.json"


In [11]:
def preprocess_vietnamese(text: str) -> List[str]:
    if not text:
        return []

    text = text.lower()
    segmented_text = word_tokenize(text, format="text")
    cleaned_text = re.sub(r'[^\w\s]', ' ', segmented_text)
    tokens = cleaned_text.split()
    return tokens


def extract_full_text(doc: Dict[str, Any]) -> str:
    parts = [
        doc.get("name", ""),
        doc.get("description", "")
    ]
    if isinstance(doc.get("comments"), list):
        parts.extend(doc.get("comments", []))

    return " ".join(parts)


In [12]:
def build_tfidf_index(docs_tokens: List[List[str]]) -> Tuple[List[Dict[str, float]], Dict[str, float]]:
    N = len(docs_tokens)

    df = defaultdict(int)
    for tokens in docs_tokens:
        for token in set(tokens):
            df[token] += 1

    idf = {}
    for token, count in df.items():
        idf[token] = math.log((1 + N) / (1 + count)) + 1

    tfidf_matrix = []
    for tokens in docs_tokens:
        if not tokens:
            tfidf_matrix.append({})
            continue

        tf_counter = Counter(tokens)
        doc_tfidf = {}
        total_words = len(tokens)

        for token, count in tf_counter.items():
            tf = count / total_words
            doc_tfidf[token] = tf * idf[token]

        tfidf_matrix.append(doc_tfidf)

    return tfidf_matrix, idf


In [13]:
def vectorize_user_query(query: str, idf_dict: Dict[str, float]) -> Dict[str, float]:
    query_tokens = preprocess_vietnamese(query)
    query_counter = Counter(query_tokens)
    total_words = len(query_tokens) if query_tokens else 1

    query_vector = {}
    for token, count in query_counter.items():
        if token in idf_dict:
            tf = count / total_words
            query_vector[token] = tf * idf_dict[token]

    return query_vector

In [14]:
def cosine_similarity(vecA: Dict[str, float], vecB: Dict[str, float]) -> float:
    dot_product = 0.0
    for token in vecA:
        if token in vecB:
            dot_product += vecA[token] * vecB[token]

    magnitudeA = math.sqrt(sum(val ** 2 for val in vecA.values()))
    magnitudeB = math.sqrt(sum(val ** 2 for val in vecB.values()))

    if magnitudeA == 0 or magnitudeB == 0:
        return 0.0

    return dot_product / (magnitudeA * magnitudeB)


def search_engine(query: str, raw_docs: List[Dict], tfidf_matrix: List[Dict], idf_dict: Dict[str, float], top_k: int = 5):
    query_vector = vectorize_user_query(query, idf_dict)

    if not query_vector:
        return

    scored_docs = []
    for idx, doc_tfidf in enumerate(tfidf_matrix):
        score = cosine_similarity(query_vector, doc_tfidf)
        if score > 0.0:
            scored_docs.append((idx, score))

    scored_docs.sort(key=lambda x: x[1], reverse=True)

    print(f"\n [Kết quả tìm kiếm cho]: '{query}'")
    print("═" * 80)

    results = scored_docs[:top_k]
    if not results:
        print("Không tìm thấy sản phẩm nào trùng khớp .")
        print("═" * 80)
        return

    for rank, (idx, score) in enumerate(results, 1):
        item = raw_docs[idx]
        print(f"Top {rank} | Cosine Score: {score:.4f}")
        print(f" Tên sản phẩm: {item.get('name')}")
        print(f" Giá bán    : {item.get('price')}")
        print(f" Liên kết   : {item.get('url')}")
        if item.get("comments"):
            print(f" Đánh giá nổi bật: \"{item['comments'][0][:120]}...\"")
        print("─" * 80)

In [15]:
if __name__ == "__main__":
    if not os.path.exists(INPUT_JSON_FILE):
        log.error(f" Không tìm thấy file '{INPUT_JSON_FILE}' trong thư mục hiện tại.")
        exit(1)

    with open(INPUT_JSON_FILE, "r", encoding="utf-8") as f:
        documents_pool = json.load(f)

    log.info(f"Đã nạp {len(documents_pool)} sản phẩm.")

    docs_text = [extract_full_text(doc) for doc in documents_pool]
    docs_tokens = [preprocess_vietnamese(text) for text in docs_text]

    tfidf_matrix, idf_dict = build_tfidf_index(docs_tokens)
    print("\n" + "="*40 + " SEARCHING ZONE " + "="*40)

    while True:
        try:
            user_input = input("\nNhập từ khóa bạn muốn tìm kiếm (hoặc gõ 'exit' để thoát): ").strip()
            if user_input.lower() == 'exit':
                print("Exit")
                break
            if not user_input:
                continue

            search_engine(user_input, documents_pool, tfidf_matrix, idf_dict, top_k=5)

        except KeyboardInterrupt:
            print("\nEnd")
            break


======================================== SEARCHING ZONE ========================================

Nhập từ khóa bạn muốn tìm kiếm (hoặc gõ 'exit' để thoát): xương khớp

 [Kết quả tìm kiếm cho]: 'xương khớp'
════════════════════════════════════════════════════════════════════════════════
Top 1 | Cosine Score: 0.4024
 Tên sản phẩm: Sữa hạt xương khớp Nutrimilk Canxi Nano 900g - Bổ sung canxi hữu cơ, hỗ trợ sức khỏe xương khớp cho mọi lứa tuổi
 Giá bán    : 329.000đ
 Liên kết   : https://chiaki.vn/sua-hat-xuong-khop-nutrimilk-canxi-nano-900g-giup-bo-sung-canxi-huu-co
 Đánh giá nổi bật: "Dùng hơn 1 tháng mà thấy xương khớp chắc khỏe hơn rõ nha XD..."
────────────────────────────────────────────────────────────────────────────────
Top 2 | Cosine Score: 0.3656
 Tên sản phẩm: Combo 3 Hộp Sữa Nutmilk Canxi Sure Gold - Sữa Hạt Dinh Dưỡng Hỗ Trợ Hệ Cơ Xương Khớp Chắc Khỏe, Bổ Sung Collagen và Glucosamin
 Giá bán    : 1.300.499đ
 Liên kết   : https://chiaki.vn/combo-3hop-sua-nutmilk-canxi-sure-go